# AI Lead Desk — Document Retrieval (RAG)

## Objective
Allow educational institutes to provide their own documents
and make the AI counselor answer using only approved information.

## Planned workflow
1. Accept institute documents such as PDF and Word files.
2. Extract their text and relevant information.
3. Split long documents into searchable chunks.
4. Create embeddings and a searchable vector index.
5. Retrieve relevant information for each student question.
6. Provide retrieved evidence to the existing LangChain counselor.
7. Refuse unsupported factual answers and offer human assistance.

## Important rules
- Each institute's information must remain isolated.
- Retrieved information must come from that institute's documents.
- Never invent fees, discounts, schedules or placement guarantees.
- Keep document sources available for verification.

In [1]:
from pathlib import Path
from pypdf import PdfReader
from docx import Document

# Locate the project root without hardcoding your Windows path.
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "requirements.txt").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "requirements.txt").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot find requirements.txt. Open JupyterLab from the "
        "AI_Lead_Desk project or its notebooks folder."
    )

# Location where institute documents will be placed for development.
DOCUMENTS_DIR = PROJECT_DIR / "data" / "institute_documents"
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

print("Document processing initialized.")
print("Project:", PROJECT_DIR)
print("Document directory:", DOCUMENTS_DIR)
print("PDF reader: ready")
print("Word reader: ready")

Document processing initialized.
Project: c:\Hehe\AI-Lead-Desk-Runtime-Errors
Document directory: c:\Hehe\AI-Lead-Desk-Runtime-Errors\data\institute_documents
PDF reader: ready
Word reader: ready


In [2]:
def extract_document_text(file_path: Path) -> str:
    """Extract text from supported institute documents."""

    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(str(file_path))
        pages = []

        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""

            if page_text.strip():
                pages.append(
                    f"[Page {page_number}]\n{page_text.strip()}"
                )

        text = "\n\n".join(pages)

    elif suffix == ".docx":
        doc = Document(str(file_path))
        sections = []

        # Extract paragraphs.
        sections.extend(
            paragraph.text.strip()
            for paragraph in doc.paragraphs
            if paragraph.text.strip()
        )

        # Extract table contents, which may contain fees or schedules.
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(
                    cell.text.strip() for cell in row.cells
                )

                if row_text.strip(" |"):
                    sections.append(row_text)

        text = "\n".join(sections)

    elif suffix == ".txt":
        text = file_path.read_text(encoding="utf-8")

    else:
        raise ValueError(
            f"Unsupported document format: {suffix}"
        )

    if not text.strip():
        raise ValueError(
            "No readable text found. The document may be scanned "
            "or empty; scanned PDFs will need OCR support."
        )

    return text.strip()


# Read all supported documents currently in the institute folder.
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt"}
documents_raw = []

document_paths = sorted(
    path for path in DOCUMENTS_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

for path in document_paths:
    try:
        extracted_text = extract_document_text(path)

        documents_raw.append({
            "source": path.name,
            "text": extracted_text
        })

        print(
            f"Read: {path.name} "
            f"({len(extracted_text):,} characters)"
        )

    except Exception as error:
        print(f"Could not read {path.name}: {error}")

if not document_paths:
    print("No documents found yet.")
    print("Place a sample PDF, DOCX or TXT file in:")
    print(DOCUMENTS_DIR)

print("\nDocuments successfully read:", len(documents_raw))

No documents found yet.
Place a sample PDF, DOCX or TXT file in:
c:\Hehe\AI-Lead-Desk-Runtime-Errors\data\institute_documents

Documents successfully read: 0


In [3]:
# Create one temporary demo document only if no supported documents exist.
existing_documents = [
    path for path in DOCUMENTS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".pdf", ".docx", ".txt"}
]

if existing_documents:
    print("Documents already exist. No demo document created.")
    print("Found:", [path.name for path in existing_documents])

else:
    demo_path = DOCUMENTS_DIR / "demo_institute_catalog.docx"

    doc = Document()
    doc.add_heading("Demo Learning Institute", level=1)
    doc.add_paragraph("DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING")

    doc.add_heading("Python Programming", level=2)
    doc.add_paragraph("Course ID: PY001")
    doc.add_paragraph("Fee: INR 8000")
    doc.add_paragraph("Duration: 3 months")
    doc.add_paragraph("Schedule: Weekends")
    doc.add_paragraph(
        "Syllabus: Python fundamentals, variables, loops, functions, "
        "lists, dictionaries and file handling."
    )

    doc.add_heading("Data Science Foundations", level=2)
    doc.add_paragraph("Course ID: DS001")
    doc.add_paragraph("Fee: INR 15000")
    doc.add_paragraph("Duration: 6 months")
    doc.add_paragraph("Schedule: Monday, Wednesday and Friday")

    doc.add_heading("Policies", level=2)
    doc.add_paragraph(
        "No approved discount or job guarantee information is available "
        "in this demo document. Refer unanswered questions to a counselor."
    )

    doc.save(demo_path)

    print("Temporary sample document created:")
    print(demo_path)
    print("\nThis is fictional test data; replace it with approved documents later.")

Temporary sample document created:
c:\Hehe\AI-Lead-Desk-Runtime-Errors\data\institute_documents\demo_institute_catalog.docx

This is fictional test data; replace it with approved documents later.


In [4]:
# Refresh the document list because the sample file was created after
# the previous extraction cell ran.

document_paths = sorted(
    path for path in DOCUMENTS_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

documents_raw = []

for path in document_paths:
    try:
        extracted_text = extract_document_text(path)

        documents_raw.append({
            "source": path.name,
            "text": extracted_text
        })

        print(f"\nDocument: {path.name}")
        print(f"Characters extracted: {len(extracted_text):,}")
        print("Preview:")
        print(extracted_text[:500])

    except Exception as error:
        print(f"Could not read {path.name}: {error}")

print("\nTotal documents processed:", len(documents_raw))



Document: demo_institute_catalog.docx
Characters extracted: 517
Preview:
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months
Schedule: Monday, Wednesday and Friday
Policies
No approved discount or job guarantee information is available in this demo document. Refer unanswered question

Total documents processed: 1


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split long documents into smaller, overlapping text sections.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

document_chunks = []

for document in documents_raw:
    chunks = text_splitter.split_text(document["text"])

    for index, chunk in enumerate(chunks):
        document_chunks.append({
            "source": document["source"],
            "chunk_id": index,
            "text": chunk
        })

print("Documents processed:", len(documents_raw))
print("Total text chunks:", len(document_chunks))

for chunk in document_chunks:
    print(f"\n--- {chunk['source']} | Chunk {chunk['chunk_id']} ---")
    print(chunk["text"])

Documents processed: 1
Total text chunks: 2

--- demo_institute_catalog.docx | Chunk 0 ---
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months

--- demo_institute_catalog.docx | Chunk 1 ---
Fee: INR 15000
Duration: 6 months
Schedule: Monday, Wednesday and Friday
Policies
No approved discount or job guarantee information is available in this demo document. Refer unanswered questions to a counselor.


In [6]:
from langchain_ollama import OllamaEmbeddings

# Use your existing local Ollama embedding model.
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text-v2-moe:latest"
)

# Extract the text from our document chunks.
chunk_texts = [
    chunk["text"]
    for chunk in document_chunks
]

if not chunk_texts:
    raise ValueError("No document chunks available to embed.")

# Convert each chunk into a numerical vector.
document_embeddings = embedding_model.embed_documents(chunk_texts)

print("Embedding model: nomic-embed-text-v2-moe:latest")
print("Chunks processed:", len(document_embeddings))
print("Vector dimensions:", len(document_embeddings[0]))
print("First vector preview:", document_embeddings[0][:5])

Embedding model: nomic-embed-text-v2-moe:latest
Chunks processed: 2
Vector dimensions: 768
First vector preview: [-0.03548131, -0.0029048282, 0.054745406, 0.023095865, -0.01502551]


In [7]:
import math

def cosine_similarity(vector_a, vector_b):
    """Measure similarity between two embedding vectors."""

    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))

    norm_a = math.sqrt(sum(a * a for a in vector_a))
    norm_b = math.sqrt(sum(b * b for b in vector_b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)


def search_documents(query: str, top_k: int = 3):
    """Return document chunks ranked by semantic similarity."""

    if not query.strip():
        return []

    # Embed the student's question using the same embedding model.
    query_embedding = embedding_model.embed_query(query)

    scored_chunks = []

    for chunk, chunk_embedding in zip(
        document_chunks, document_embeddings
    ):
        score = cosine_similarity(query_embedding, chunk_embedding)

        scored_chunks.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "similarity": score
        })

    # Highest similarity first.
    scored_chunks.sort(
        key=lambda item: item["similarity"],
        reverse=True
    )

    return scored_chunks[:top_k]


# Test semantic search with wording different from the document.
test_query = "How much do I need to pay for Python classes?"

search_results = search_documents(test_query)

print("Student question:", test_query)

for result in search_results:
    print(f"\nSource: {result['source']}")
    print(f"Chunk: {result['chunk_id']}")
    print(f"Similarity: {result['similarity']:.4f}")
    print("Retrieved text:")
    print(result["text"])

Student question: How much do I need to pay for Python classes?

Source: demo_institute_catalog.docx
Chunk: 0
Similarity: 0.5178
Retrieved text:
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months

Source: demo_institute_catalog.docx
Chunk: 1
Similarity: 0.2541
Retrieved text:
Fee: INR 15000
Duration: 6 months
Schedule: Monday, Wednesday and Friday
Policies
No approved discount or job guarantee information is available in this demo document. Refer unanswered questions to a counselor.


In [8]:
def search_documents(
    query: str,
    top_k: int = 3,
    min_similarity: float = 0.30
):
    """Retrieve document chunks only when they meet a relevance threshold."""

    if not query.strip():
        return []

    query_embedding = embedding_model.embed_query(query)
    scored_chunks = []

    for chunk, chunk_embedding in zip(
        document_chunks, document_embeddings
    ):
        score = cosine_similarity(query_embedding, chunk_embedding)

        # Reject chunks below our initial demo threshold.
        if score < min_similarity:
            continue

        scored_chunks.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "similarity": score
        })

    scored_chunks.sort(
        key=lambda item: item["similarity"],
        reverse=True
    )

    return scored_chunks[:top_k]


# Test a known question and an unrelated question.
for query in [
    "How much do I need to pay for Python classes?",
    "Does the institute teach underwater basket weaving?"
]:
    results = search_documents(query)

    print(f"\nQuestion: {query}")

    if not results:
        print("No sufficiently relevant document chunks found.")
    else:
        for result in results:
            print(
                f"Source: {result['source']} | "
                f"Similarity: {result['similarity']:.4f}"
            )
            print(result["text"][:250])


Question: How much do I need to pay for Python classes?
Source: demo_institute_catalog.docx | Similarity: 0.5178
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and 

Question: Does the institute teach underwater basket weaving?
No sufficiently relevant document chunks found.


In [9]:
def get_retrieved_context(
    query: str,
    top_k: int = 3,
    min_similarity: float = 0.30
) -> dict:
    """
    Retrieve relevant institute document chunks and preserve their sources.
    Returns an empty context when no relevant information is found.
    """

    results = search_documents(
        query=query,
        top_k=top_k,
        min_similarity=min_similarity
    )

    if not results:
        return {
            "context": "",
            "sources": [],
            "retrieved_count": 0
        }

    context_parts = []
    sources = []

    for result in results:
        source_label = (
            f"{result['source']} | chunk {result['chunk_id']}"
        )

        context_parts.append(
            f"Source: {source_label}\n"
            f"Document text:\n{result['text']}"
        )

        sources.append({
            "source": result["source"],
            "chunk_id": result["chunk_id"],
            "similarity": round(result["similarity"], 4)
        })

    return {
        "context": "\n\n---\n\n".join(context_parts),
        "sources": sources,
        "retrieved_count": len(results)
    }


# Test with a question supported by our demo document.
query = "How much does Python Programming cost?"

retrieval = get_retrieved_context(query)

print("Question:", query)
print("Retrieved chunks:", retrieval["retrieved_count"])
print("\nSources:", retrieval["sources"])
print("\nContext for the LLM:")
print(retrieval["context"] or "No relevant context found.")

Question: How much does Python Programming cost?
Retrieved chunks: 1

Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.5001}]

Context for the LLM:
Source: demo_institute_catalog.docx | chunk 0
Document text:
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months


In [10]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load the existing project credentials.
load_dotenv(PROJECT_DIR / ".env")

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY not found. Check the .env file in the project root."
    )

# Use the same model selected for AI Lead Desk.
rag_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=512,
    max_retries=2
)

print("RAG answer-generation model initialized.")
print("Model: openai/gpt-oss-120b")
print("Temperature: 0")

RAG answer-generation model initialized.
Model: openai/gpt-oss-120b
Temperature: 0


In [12]:
from langchain_core.messages import SystemMessage, HumanMessage


def answer_from_documents(question: str) -> dict:
    """
    Retrieve institute document evidence and generate an answer
    using only the retrieved context.
    """

    # 1. Retrieve relevant document chunks.
    retrieval = get_retrieved_context(question)

    # 2. Do not call the LLM if retrieval finds no relevant evidence.
    if not retrieval["context"]:
        return {
            "answer": (
                "I couldn't verify this information from the "
                "institute's documents. Would you like to speak "
                "with an admissions counselor?"
            ),
            "sources": [],
            "grounded": False
        }

    # 3. Give the LLM only the retrieved evidence for factual answers.
    system_prompt = """
You are an AI admissions counselor.

STRICT EVIDENCE RULES:
- Treat the supplied document excerpts as your only source of facts.
- Do not add facts from your pretrained knowledge.
- Do not invent fees, discounts, schedules, policies or placement claims.
- If the excerpts do not explicitly support the answer, say that the
  information cannot be verified and offer human assistance.
- Do not treat missing information as proof of a policy.
- Reply in the language and conversational style of the student's question.
- Be concise and do not claim that an external action was completed.

DOCUMENT EXCERPTS:
{context}
"""

    messages = [
        SystemMessage(
            content=system_prompt.format(
                context=retrieval["context"]
            )
        ),
        HumanMessage(content=question)
    ]

    # 4. Generate a real response through the Groq API.
    response = rag_llm.invoke(messages)

    return {
        "answer": str(response.content).strip(),
        "sources": retrieval["sources"],
        "grounded": True
    }


# Test with a question answered by the demo document.
question = "How much does Python Programming cost?"

result = answer_from_documents(question)

print("Student:", question)
print("\nAI Counselor:", result["answer"])
print("\nRetrieved sources:", result["sources"])
print("Evidence retrieved:", result["grounded"])

Student: How much does Python Programming cost?

AI Counselor: The Python Programming course costs **INR 8,000**.

Retrieved sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.5001}]
Evidence retrieved: True


In [13]:
test_questions = [
    "What is the fee for Python Programming?",
    "Does Python Programming guarantee a job?",
    "What is the fee for the Robotics course?",
    "Do you teach underwater basket weaving?"
]

for question in test_questions:
    print("\n" + "=" * 65)
    print("STUDENT:", question)

    result = answer_from_documents(question)

    print("\nAI COUNSELOR:", result["answer"])
    print("\nEvidence retrieved:", result["grounded"])
    print("Sources:", result["sources"])


STUDENT: What is the fee for Python Programming?

AI COUNSELOR: The fee for the **Python Programming** course is **INR 8,000**.

Evidence retrieved: True
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.553}]

STUDENT: Does Python Programming guarantee a job?

AI COUNSELOR: The provided information about the Python Programming course does not mention any job‑placement guarantee. If you’d like more details, I can connect you with a human admissions advisor.

Evidence retrieved: True
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.3872}]

STUDENT: What is the fee for the Robotics course?

AI COUNSELOR: I’m sorry, but the provided information does not include any details about a Robotics course, so I can’t confirm its fee. If you’d like, I can connect you with a counselor who can help you further.

Evidence retrieved: True
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.3756}, {'source

In [14]:
# Use a stricter relevance threshold for the RAG answer pipeline.
# This replaces the earlier answer_from_documents function.

RAG_MIN_SIMILARITY = 0.42

def answer_from_documents(question: str) -> dict:
    """Generate an answer only when document retrieval is sufficiently relevant."""

    retrieval = get_retrieved_context(
        query=question,
        top_k=3,
        min_similarity=RAG_MIN_SIMILARITY
    )

    # Do not call the LLM when retrieval finds no sufficiently relevant evidence.
    if not retrieval["context"]:
        return {
            "answer": (
                "I couldn't verify this information from the institute's "
                "documents. Would you like to speak with an admissions counselor?"
            ),
            "sources": [],
            "grounded": False
        }

    system_prompt = """
You are an AI admissions counselor.

Use only facts explicitly supported by the supplied document excerpts.
Never invent fees, discounts, schedules, courses or placement guarantees.
Do not infer that a course or policy exists merely because it is mentioned
in an unrelated excerpt.
If the excerpts do not answer the question, say the information cannot
be verified and offer human assistance.
Reply in the language of the student's latest message.

DOCUMENT EXCERPTS:
{context}
"""

    messages = [
        SystemMessage(
            content=system_prompt.format(context=retrieval["context"])
        ),
        HumanMessage(content=question)
    ]

    response = rag_llm.invoke(messages)

    return {
        "answer": str(response.content).strip(),
        "sources": retrieval["sources"],
        "grounded": True
    }


# Test a known course and an unknown course.
for question in [
    "How much does Python Programming cost?",
    "What is the fee for the Robotics course?"
]:
    result = answer_from_documents(question)

    print("\nQuestion:", question)
    print("Answer:", result["answer"])
    print("Evidence retrieved:", result["grounded"])
    print("Sources:", result["sources"])


Question: How much does Python Programming cost?
Answer: The Python Programming course costs **INR 8,000**.
Evidence retrieved: True
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 0, 'similarity': 0.5001}]

Question: What is the fee for the Robotics course?
Answer: I couldn't verify this information from the institute's documents. Would you like to speak with an admissions counselor?
Evidence retrieved: False
Sources: []


In [15]:
from docx.text.paragraph import Paragraph
from docx.table import Table
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Keep related information together, while splitting long sections.
section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

document_chunks = []

for document in documents_raw:
    source = document["source"]
    file_path = DOCUMENTS_DIR / source
    sections = []

    if file_path.suffix.lower() == ".docx":
        doc = Document(str(file_path))

        current_title = "Document introduction"
        current_lines = []

        def save_section():
            if current_lines:
                sections.append({
                    "title": current_title,
                    "text": "\n".join(current_lines).strip()
                })

        # Preserve paragraph headings and table content in document order.
        for block in doc.element.body.iterchildren():

            if isinstance(block, CT_P):
                paragraph = Paragraph(block, doc)
                text = paragraph.text.strip()

                if not text:
                    continue

                style_name = paragraph.style.name.lower()

                if style_name.startswith("heading"):
                    save_section()
                    current_title = text
                    current_lines = [text]
                else:
                    current_lines.append(text)

            elif isinstance(block, CT_Tbl):
                table = Table(block, doc)

                for row in table.rows:
                    row_text = " | ".join(
                        cell.text.strip() for cell in row.cells
                    )

                    if row_text.strip(" |"):
                        current_lines.append(row_text)

        save_section()

    else:
        # Generic chunking for PDFs and text files.
        sections = [{
            "title": "Document text",
            "text": document["text"]
        }]

    # Split each section independently.
    for section in sections:
        parts = section_splitter.split_text(section["text"])

        for part in parts:
            if part.strip():
                document_chunks.append({
                    "source": source,
                    "section": section["title"],
                    "chunk_id": len(document_chunks),
                    "text": part
                })

# Recreate embeddings because the chunks have changed.
chunk_texts = [chunk["text"] for chunk in document_chunks]

if not chunk_texts:
    raise ValueError("No searchable document chunks were created.")

document_embeddings = embedding_model.embed_documents(chunk_texts)

print("Documents:", len(documents_raw))
print("Section-aware chunks:", len(document_chunks))
print("Embeddings generated:", len(document_embeddings))

for chunk in document_chunks:
    print(f"\n--- {chunk['source']} | {chunk['section']} ---")
    print(chunk["text"])

Documents: 1
Section-aware chunks: 4
Embeddings generated: 4

--- demo_institute_catalog.docx | Demo Learning Institute ---
Demo Learning Institute
DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING

--- demo_institute_catalog.docx | Python Programming ---
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.

--- demo_institute_catalog.docx | Data Science Foundations ---
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months
Schedule: Monday, Wednesday and Friday

--- demo_institute_catalog.docx | Policies ---
Policies
No approved discount or job guarantee information is available in this demo document. Refer unanswered questions to a counselor.


In [16]:
# Test retrieval after section-aware chunking.

test_questions = [
    "What is the fee for Python Programming?",
    "What is the fee for Data Science Foundations?",
    "Does Python Programming guarantee a job?",
    "What is the fee for Robotics?"
]

for question in test_questions:
    retrieval = get_retrieved_context(
        query=question,
        top_k=2,
        min_similarity=RAG_MIN_SIMILARITY
    )

    print("\n" + "=" * 60)
    print("Question:", question)
    print("Retrieved chunks:", retrieval["retrieved_count"])
    print("Sources:", retrieval["sources"])
    print("\nRetrieved evidence:")
    print(retrieval["context"] or "No relevant evidence found.")


Question: What is the fee for Python Programming?
Retrieved chunks: 1
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.6258}]

Retrieved evidence:
Source: demo_institute_catalog.docx | chunk 1
Document text:
Python Programming
Course ID: PY001
Fee: INR 8000
Duration: 3 months
Schedule: Weekends
Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file handling.

Question: What is the fee for Data Science Foundations?
Retrieved chunks: 1
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 2, 'similarity': 0.5939}]

Retrieved evidence:
Source: demo_institute_catalog.docx | chunk 2
Document text:
Data Science Foundations
Course ID: DS001
Fee: INR 15000
Duration: 6 months
Schedule: Monday, Wednesday and Friday

Question: Does Python Programming guarantee a job?
Retrieved chunks: 1
Sources: [{'source': 'demo_institute_catalog.docx', 'chunk_id': 1, 'similarity': 0.4882}]

Retrieved evidence:
Source: demo_institute_

In [17]:
## Check the Retrival Logic

# Diagnose why policy questions are retrieving course chunks
# instead of the Policies section.

diagnostic_questions = [
    "Does Python Programming guarantee a job?",
    "What is the institute's job placement policy?",
    "Are discounts or scholarships available?"
]

for question in diagnostic_questions:
    query_embedding = embedding_model.embed_query(question)
    scored_chunks = []

    for chunk, chunk_embedding in zip(
        document_chunks,
        document_embeddings
    ):
        similarity = cosine_similarity(
            query_embedding,
            chunk_embedding
        )

        scored_chunks.append({
            "section": chunk.get("section", "Unknown"),
            "chunk_id": chunk["chunk_id"],
            "similarity": round(float(similarity), 4),
            "text": chunk["text"][:180].replace("\n", " ")
        })

    scored_chunks.sort(
        key=lambda item: item["similarity"],
        reverse=True
    )

    print("\n" + "=" * 75)
    print("Question:", question)

    for result in scored_chunks:
        print(
            f"\nSection: {result['section']} | "
            f"Chunk: {result['chunk_id']} | "
            f"Similarity: {result['similarity']}"
        )
        print("Text:", result["text"])


Question: Does Python Programming guarantee a job?

Section: Python Programming | Chunk: 1 | Similarity: 0.4882
Text: Python Programming Course ID: PY001 Fee: INR 8000 Duration: 3 months Schedule: Weekends Syllabus: Python fundamentals, variables, loops, functions, lists, dictionaries and file han

Section: Policies | Chunk: 3 | Similarity: 0.3855
Text: Policies No approved discount or job guarantee information is available in this demo document. Refer unanswered questions to a counselor.

Section: Data Science Foundations | Chunk: 2 | Similarity: 0.1683
Text: Data Science Foundations Course ID: DS001 Fee: INR 15000 Duration: 6 months Schedule: Monday, Wednesday and Friday

Section: Demo Learning Institute | Chunk: 0 | Similarity: 0.1335
Text: Demo Learning Institute DEMO DATA — NOT APPROVED FOR REAL STUDENT COUNSELLING

Question: What is the institute's job placement policy?

Section: Policies | Chunk: 3 | Similarity: 0.306
Text: Policies No approved discount or job guarantee informa

In [18]:
# Cell 18: Policy-aware document retrieval

POLICY_QUERY_TERMS = (
    "guarantee",
    "guaranteed job",
    "job guarantee",
    "placement policy",
    "placement guarantee",
    "discount",
    "scholarship",
    "special offer",
    "discount policy",
    "policy",
    "नोकरीची हमी",
    "नोकरीची",
    "प्लेसमेंट",
    "सवलत",
    "छूट",
    "स्कॉलरशिप",
)

POLICY_EVIDENCE_TERMS = (
    "policy",
    "policies",
    "guarantee",
    "placement",
    "discount",
    "scholarship",
    "offer",
    "नोकरी",
    "प्लेसमेंट",
    "सवलत",
    "छूट",
)

POLICY_MIN_SIMILARITY = 0.28


def search_documents(query, top_k=3, min_similarity=0.30):
    """
    Search embedded document chunks.

    Policy-related questions use a policy-specific evidence filter
    and threshold. Other questions retain their supplied threshold.
    """
    query_embedding = embedding_model.embed_query(query)
    query_lower = query.lower()

    is_policy_query = any(
        term in query_lower
        for term in POLICY_QUERY_TERMS
    )

    scored_chunks = []

    for chunk, chunk_embedding in zip(
        document_chunks,
        document_embeddings
    ):
        similarity = cosine_similarity(
            query_embedding,
            chunk_embedding
        )

        scored_chunks.append({
            "source": chunk["source"],
            "section": chunk.get("section", "Unknown"),
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "similarity": float(similarity),
        })

    # For policy questions, prefer evidence that discusses policy topics.
    if is_policy_query:
        policy_candidates = [
            item
            for item in scored_chunks
            if any(
                term in (
                    str(item["section"]).lower()
                    + " "
                    + item["text"].lower()
                )
                for term in POLICY_EVIDENCE_TERMS
            )
        ]

        # Do not silently fall back to unrelated course chunks
        # when no policy-related evidence exists.
        scored_chunks = policy_candidates
        effective_threshold = min(
            min_similarity,
            POLICY_MIN_SIMILARITY
        )
    else:
        effective_threshold = min_similarity

    results = [
        item
        for item in scored_chunks
        if item["similarity"] >= effective_threshold
    ]

    results.sort(
        key=lambda item: item["similarity"],
        reverse=True
    )

    return results[:top_k]


print("Policy-aware retrieval is ready.")
print("Normal query threshold: configured by caller")
print("Policy query threshold:", POLICY_MIN_SIMILARITY)

Policy-aware retrieval is ready.
Normal query threshold: configured by caller
Policy query threshold: 0.28
